# Day 36: Model and Data Version Documentation

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Save data provenance, model hashes, original split indices and environment versions.

## Theory

Reproducibility requires identifying the exact data, split, preprocessing and model bytes. A manifest connects those artifacts to the evaluation while preserving the existing fitted estimator.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Generate the manifest and explicit train/test row indices.

In [2]:
from src.versioning import save_manifest
m=save_manifest()
print(json.dumps({"dataset":{k:m["dataset"][k] for k in ["rows","train_rows","test_rows","cached_csv_sha256"]},"model":m["model"],"environment":m["environment"]},indent=2))

{
  "dataset": {
    "rows": 649,
    "train_rows": 519,
    "test_rows": 130,
    "cached_csv_sha256": "5be7b02f13f5800fddd66e2c8d834cd00ece14c552fb404def2397b31b63236b"
  },
  "model": {
    "sha256": "1bedf35d0d56c4f090cd28d2a317a10533eba0e28489d2318aebdde2d105fb83",
    "estimator": "DecisionTreeClassifier",
    "parameters": {
      "ccp_alpha": 0.0,
      "class_weight": null,
      "criterion": "gini",
      "max_depth": 10,
      "max_features": null,
      "max_leaf_nodes": null,
      "min_impurity_decrease": 0.0,
      "min_samples_leaf": 1,
      "min_samples_split": 10,
      "min_weight_fraction_leaf": 0.0,
      "monotonic_cst": null,
      "random_state": 42,
      "splitter": "best"
    },
    "selection": "training-only five-fold stratified CV Macro F1",
    "classes": [
      "High",
      "Low",
      "Medium"
    ]
  },
  "environment": {
    "python": "3.14.2",
    "numpy": "2.5.4",
    "pandas": "3.1.0",
    "scikit-learn": "1.9.0",
    "joblib": "1.6.0",
    "st

Verify split separation and saved artifact consistency.

In [3]:
indices=json.loads(Path("artifacts/split_indices.json").read_text())
assert not set(indices["train"]) & set(indices["test"])
assert len(set(indices["train"]+indices["test"]))==649
assert "G3" not in m["dataset"]["features"]
assert m["model"]["sha256"]==Path("artifacts/student_performance_pipeline.sha256").read_text().strip()
print("Split separation, row coverage, target exclusion and model checksum: PASS")

Split separation, row coverage, target exclusion and model checksum: PASS


## Findings and interpretation

The manifest identifies the public data and trained model exactly. All 649 original row indices are covered once across disjoint training and evaluation partitions.

## Limitations

Matching hashes establishes file consistency, not scientific validity or trusted origin. The manifest documents historical evaluation reuse.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Save data provenance, model hashes, original split indices and environment versions.

## Next-day plan

Expand automated regression coverage and continuous integration.